# Redes neuronales para clasificación de imagenes
---

- **Taller 1**: el perceptron multicapa
- **Fecha de entrega**: 13 de Septiembre de 2024.


Importamos librerías:

In [ ]:
import numpy as np
import pandas as pd
import keras
import tensorflow as tf
import matplotlib.pyplot as plt
from keras.callbacks import EarlyStopping
from sklearn.manifold import TSNE


#1. Cargue el conjunto de datos MNIST

Use KERAS para cargar el conjunto de datos MNIST (consulte: https://keras.io/api/datasets/mnist/), una vez cargado añada ruido aleatorio, tanto en el conjunto de entrenamiento como el de validación. Deberá obtener cuatro conjuntos: entrenamiento+ruido, test+ruido, entrenamiento simple y test simple.

Muestre un ejemplo para cada una de las etiquetas con y sin ruido (Tómelos del conjunto de entrenamiento).

In [ ]:

from tensorflow.keras.datasets import mnist

#Carga el conjunto de datos MNIST y normaliza los valores de los píxeles.
#Returns: Datos de entrenamiento y prueba normalizados.

def cargar_datos():

    (x_ent, y_ent), (x_pru, y_pru) = mnist.load_data()

    x_ent = x_ent.astype('float32') / 255.0
    x_pru = x_pru.astype('float32') / 255.0

    return (x_ent, y_ent), (x_pru, y_pru)


#Añade ruido gaussiano a los datos y recorta los valores al rango [0, 1].
#Args: datos (np.array): Datos de entrada. factor (float): Intensidad del ruido a añadir.
#  Returns:np.array: Datos con ruido añadido.

def añadir_ruido(datos, factor=0.5):

    ruido = factor * np.random.normal(loc=0.0, scale=1.0, size=datos.shape)
    return np.clip(datos + ruido, 0., 1.)


# Visualiza ejemplos de imágenes originales y con ruido para cada clase.
#x_orig (np.array): Datos originales.  x_ruido (np.array): Datos con ruido.
#y_datos (np.array): Etiquetas de los datos.  num_clases (int): Número de clases a visualizar.

def graficar(x_orig, x_ruido, y_datos, num_clases=10):
    plt.figure(figsize=(20, 4))
    for i in range(num_clases):
        # Imagen original
        plt.subplot(2, num_clases, i + 1)
        plt.imshow(x_orig[y_datos == i][0], cmap='gray')
        plt.title(f"Orig: {i}")
        plt.axis('off')

        # Imagen con ruido
        plt.subplot(2, num_clases, i + 1 + num_clases)
        plt.imshow(x_ruido[y_datos == i][0], cmap='gray')
        plt.title(f"Ruido: {i}")
        plt.axis('off')

    plt.tight_layout()
    plt.show()

def main():
    # Cargar datos
    (x_ent, y_ent), (x_pru, y_pru) = cargar_datos()

    # Añadir ruido
    x_ent_ruido = añadir_ruido(x_ent)
    x_pru_ruido = añadir_ruido(x_pru)

    # Visualizar ejemplos
    graficar(x_ent, x_ent_ruido, y_ent)

if __name__ == "__main__":
    main()

# 2. Proponga una estructura neuronal

En este caso, construirá una red neuronal que sea capaz de limpiar nuestras imágenes. Por lo cual, los conjuntos con ruido serán los datos de entrada y las imágenes limpias la salida. Tomando esto en consideración, proponga una estructura neuronal capaz de realizar dicha tarea.

**Pistas:**


1.   Recuerde aplanar las imágenes usando reshape o una capa de flatten.
2.   Recuerde que la entrada como la salida serán vectores de 784.
3.   La capa intermedia deberá tener menos neuronas que la de entrada y salida.
4.   Piense si esto es una tarea de clasificación (loss: binary cross entropy, metric: accuracy) o de regresión (loss: Mean squared error, metric: mean absolute error) para que pueda seleccionar adecuadamente la función de coste.





In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

def crear_autoencoder_mejorado(
    entrada_shape=(784,),
    capa_intermedia=[512, 256, 128],
    activacion_intermedia='relu',
    activacion_salida='sigmoid',
    optimizador='rmsprop',
    perdida='binary_crossentropy'
):

    entrada = layers.Input(shape=entrada_shape)

    codificador = entrada
    for capa in capa_intermedia:
        codificador = layers.Dense(capa, activation=activacion_intermedia,
                                   kernel_regularizer=tf.keras.regularizers.l2(0.01))(codificador)
        codificador = layers.BatchNormalization()(codificador)
        codificador = layers.Dropout(0.4)(codificador)

    decodificador = codificador
    for capa in reversed(capa_intermedia):
        decodificador = layers.Dense(capa, activation=activacion_intermedia)(decodificador)
        decodificador = layers.BatchNormalization()(decodificador)
        decodificador = layers.Dropout(0.4)(decodificador)

    salida = layers.Dense(entrada_shape[0], activation=activacion_salida)(decodificador)

    autoencoder = models.Model(inputs=entrada, outputs=salida, name="Autoencoder")

    autoencoder.compile(
        optimizer=tf.keras.optimizers.RMSprop(learning_rate=0.0005),
        loss=perdida,
        metrics=['mae', 'mse']
    )

    return autoencoder


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
from tensorflow.keras.datasets import mnist

def crear_autoencoder(
    entrada_shape=(784,),
    capa_intermedia=[512, 256, 128],
    activacion_intermedia='relu',
    activacion_salida='sigmoid',
    optimizador='adamw',
    perdida='binary_crossentropy'
):

    entrada = layers.Input(shape=entrada_shape)

    # Capas codificadoras con Batch Normalization y Dropout
    codificador = entrada
    for capa in capa_intermedia:
        codificador = layers.Dense(capa, activation=activacion_intermedia,
                                   kernel_regularizer=tf.keras.regularizers.l2(0.01))(codificador)
        codificador = layers.BatchNormalization()(codificador)
        codificador = layers.Dropout(0.4)(codificador)

    # Capas decodificadoras
    decodificador = codificador
    for capa in reversed(capa_intermedia):
        decodificador = layers.Dense(capa, activation=activacion_intermedia)(decodificador)
        decodificador = layers.BatchNormalization()(decodificador)
        decodificador = layers.Dropout(0.4)(decodificador)

    salida = layers.Dense(entrada_shape[0], activation=activacion_salida)(decodificador)

    autoencoder = models.Model(inputs=entrada, outputs=salida, name="Autoencoder")

    # Configurar optimizador AdamW
    autoencoder.compile(
        optimizer=tf.keras.optimizers.AdamW(learning_rate=0.0005, weight_decay=1e-4),
        loss=perdida,
        metrics=['mae', 'mse']
    )

    return autoencoder

def preparar_datos(x_train, x_test, factor_ruido=0.5):
    def añadir_ruido(x, factor):
        ruido = np.random.normal(loc=0.0, scale=1.0, size=x.shape)
        x_ruido = x + factor * ruido
        return np.clip(x_ruido, 0., 1.)

    x_train_ruido = añadir_ruido(x_train, factor_ruido)
    x_test_ruido = añadir_ruido(x_test, factor_ruido)

    x_train_reshaped = x_train.reshape(-1, 784)
    x_train_ruido_reshaped = x_train_ruido.reshape(-1, 784)
    x_test_reshaped = x_test.reshape(-1, 784)
    x_test_ruido_reshaped = x_test_ruido.reshape(-1, 784)

    return x_train_ruido_reshaped, x_train_reshaped, x_test_ruido_reshaped, x_test_reshaped

def entrenar_modelo(modelo, x_train, y_train, x_val, y_val, epochs=50, batch_size=256):
    early_stopping = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)
    reduce_lr = ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=5, min_lr=0.001)
    historial = modelo.fit(
        x_train,
        y_train,
        epochs=epochs,
        batch_size=batch_size,
        shuffle=True,
        validation_data=(x_val, y_val),
        callbacks=[early_stopping, reduce_lr]
    )
    return historial

def main():
    # Cargar el conjunto de datos MNIST
    (x_train, _), (x_test, _) = mnist.load_data()

    # Normalizar los valores de los píxeles para estar entre 0 y 1
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0

    # Preparar los datos
    x_train_ruido, x_train_limpio, x_test_ruido, x_test_limpio = preparar_datos(x_train, x_test)

    # Crear el modelo autoencoder
    modelo_autoencoder = crear_autoencoder()

    # Entrenar el modelo
    entrenar_modelo(modelo_autoencoder, x_train_ruido, x_train_limpio, x_test_ruido, x_test_limpio)

if __name__ == "__main__":
    main()



# 3. Entrena la red neuronal y demuestre su desempeño

Entre la red neuronal y demuestre el desempeñó. Justifique su elección tanto de función de perdida como de métrica de evaluación. Adicionalmente, muestre los resultados obtenidos para el conjunto de test (recuerde convertir su vector de 784 a una imagen de 28x28).

In [ ]:
import matplotlib.pyplot as plt

def evaluar_modelo(modelo, x_test_ruido, x_test_limpio):
    # Evaluación en el conjunto de prueba
    resultado = modelo.evaluate(x_test_ruido, x_test_limpio, verbose=0)
    print(f"Pérdida en el conjunto de prueba (MSE): {resultado[0]:.4f}")
    print(f"Error absoluto medio en el conjunto de prueba (MAE): {resultado[1]:.4f}")

def hacer_predicciones(modelo, x_test_ruido):
    return modelo.predict(x_test_ruido)

def mostrar_resultados(x_original, x_ruido, x_pred, num_imagenes=10):
    plt.figure(figsize=(20, 6))
    for i in range(num_imagenes):
        # Mostrar imagen original
        ax = plt.subplot(3, num_imagenes, i + 1)
        plt.imshow(x_original[i].reshape(28, 28), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        if i == num_imagenes // 2:
            ax.set_title("Original")

        # Mostrar imagen con ruido
        ax = plt.subplot(3, num_imagenes, i + 1 + num_imagenes)
        plt.imshow(x_ruido[i].reshape(28, 28), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        if i == num_imagenes // 2:
            ax.set_title("Con Ruido")

        # Mostrar imagen reconstruida
        ax = plt.subplot(3, num_imagenes, i + 1 + 2 * num_imagenes)
        plt.imshow(x_pred[i].reshape(28, 28), cmap='gray')
        ax.get_xaxis().set_visible(False)
        ax.get_yaxis().set_visible(False)
        if i == num_imagenes // 2:
            ax.set_title("Reconstruida")

    plt.subplots_adjust(wspace=0.1, hspace=0.5)
    plt.show()

def main():
    # Cargar el conjunto de datos MNIST
    (x_train, _), (x_test, _) = mnist.load_data()

    # Normalizar los valores de los píxeles para estar entre 0 y 1
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0

    # Preparar los datos
    x_train_ruido, x_train_limpio, x_test_ruido, x_test_limpio = preparar_datos(x_train, x_test)

    # Crear el modelo autoencoder
    modelo_autoencoder = crear_autoencoder()

    # Entrenar el modelo
    entrenar_modelo(modelo_autoencoder, x_train_ruido, x_train_limpio, x_test_ruido, x_test_limpio)

    # Evaluar el modelo
    evaluar_modelo(modelo_autoencoder, x_test_ruido, x_test_limpio)

    # Realizar predicciones
    predicciones = hacer_predicciones(modelo_autoencoder, x_test_ruido)

    # Visualizar resultados
    mostrar_resultados(x_test_limpio, x_test_ruido, predicciones)

if __name__ == "__main__":
    main()


# 4. Extraiga la representación

Recuerde que la capa intermedia DEBE tener un tamaño menor a las capas de entrada y de salida. Extraiga los valores de dicha capa para cada una de las imágenes. Use una técnica de reducción de dimensionalidad (consulte: https://scikit-learn.org/stable/modules/generated/sklearn.manifold.TSNE.html) y observe un diagrama de dispersión de la representación neuronal etiquetado con cada una de las clases del conjunto de datos.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.manifold import TSNE
from tensorflow.keras.datasets import mnist
from tensorflow.keras import models

def extraer_representacion(modelo, x_test_ruido):
    # Obtener la capa intermedia del modelo
    capa_intermedia = modelo.layers[3]  # Ajustar el índice según la estructura del modelo

    # Crear un nuevo modelo que tome la entrada original y produzca la salida de la capa intermedia
    modelo_intermedio = models.Model(inputs=modelo.input, outputs=capa_intermedia.output)

    # Obtener la representación de la capa intermedia para los datos de prueba
    representacion = modelo_intermedio.predict(x_test_ruido)

    return representacion

def reducir_dimensionalidad_tsne(representacion, n_components=2):
    tsne = TSNE(n_components=n_components, random_state=42)
    representacion_reducida = tsne.fit_transform(representacion)
    return representacion_reducida

def visualizar_representacion(representacion_reducida, y_test, num_clases=10):
    # Usar la paleta 'tab10' de matplotlib para 10 clases
    colores = plt.get_cmap('tab10')

    plt.figure(figsize=(10, 8))
    for i in range(num_clases):
        indices = y_test == i
        plt.scatter(representacion_reducida[indices, 0], representacion_reducida[indices, 1],
                    color=colores(i), label=str(i), alpha=0.7, edgecolors='w', linewidth=0.5)

    plt.legend()
    plt.title("Representación Neuronal con TSNE")
    plt.xlabel("Componente 1")
    plt.ylabel("Componente 2")
    plt.show()

def main():
    # Cargar el conjunto de datos MNIST
    (x_train, y_train), (x_test, y_test) = mnist.load_data()

    # Normalizar los valores de los píxeles para estar entre 0 y 1
    x_train = x_train.astype('float32') / 255.0
    x_test = x_test.astype('float32') / 255.0

    # Preparar los datos
    x_train_ruido, x_train_limpio, x_test_ruido, x_test_limpio = preparar_datos(x_train, x_test)

    # Crear el modelo autoencoder
    modelo_autoencoder = crear_autoencoder()

    # Entrenar el modelo
    entrenar_modelo(modelo_autoencoder, x_train_ruido, x_train_limpio, x_test_ruido, x_test_limpio)

    # Extraer la representación de la capa intermedia
    representacion = extraer_representacion(modelo_autoencoder, x_test_ruido)

    # Reducir la dimensionalidad con TSNE
    representacion_reducida = reducir_dimensionalidad_tsne(representacion)

    # Visualizar la representación
    visualizar_representacion(representacion_reducida, y_test)

if __name__ == "__main__":
    main()

